# Cycle 3 — Tuning (Chronological Split)

Same `RandomizedSearchCV` configuration as `notebooks/cycle3_tuning.ipynb`. Train/test split is by `start_year`. Inner CV is 5-fold stratified on the earlier seasons (training partition).

## Setup & data (chronological by start_year)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier
import joblib, os

raw = pd.read_csv('../data/raw/player_injuries.csv')
raw['High_Injury'] = (raw['season_days_injured']>=28).astype(int)
wr_map={'Low':1,'Medium':2,'High':3}; pos_map={'GK':1,'DF':2,'MF':3,'FW':4}
if 'work_rate' in raw.columns:
    raw['work_rate_numeric'] = raw['work_rate'].astype(str).str.split('/').str[0].str.strip().map(wr_map).fillna(2)
if 'position' in raw.columns:
    raw['position_numeric'] = raw['position'].astype(str).str[:2].map(pos_map).fillna(3)
if 'height_cm' in raw.columns and 'weight_kg' in raw.columns:
    raw['bmi'] = raw['weight_kg'] / (raw['height_cm']/100.0)**2

drop_cols = ['p_id2','dob','nationality','work_rate','position',
             'season_days_injured','total_days_injured',
             'season_minutes_played','season_games_played','season_matches_in_squad',
             'total_minutes_played','total_games_played']
df = raw.drop(columns=[c for c in drop_cols if c in raw.columns])
history_cols = ['cumulative_minutes_played','cumulative_games_played',
                'minutes_per_game_prev_seasons','avg_days_injured_prev_seasons',
                'avg_games_per_season_prev_seasons','significant_injury_prev_season',
                'cumulative_days_injured','season_days_injured_prev_season']
history_cols = [c for c in history_cols if c in df.columns]
df[history_cols] = df[history_cols].fillna(0)
df = df.dropna().sort_values('start_year').reset_index(drop=True)

split_idx = int(len(df)*0.8)
train_df = df.iloc[:split_idx]; test_df = df.iloc[split_idx:]
drop_for_X = ['High_Injury','start_year']
X_train = train_df.drop(columns=drop_for_X); y_train = train_df['High_Injury']
X_test  = test_df.drop(columns=drop_for_X);  y_test  = test_df['High_Injury']

fc_path = '../models/cycle3_feature_cols.pkl'
if os.path.exists(fc_path):
    expected = joblib.load(fc_path)
    extras = [c for c in X_train.columns if c not in expected]
    if extras: X_train = X_train.drop(columns=extras); X_test = X_test.drop(columns=extras)
    missing = [c for c in expected if c not in X_train.columns]
    for m in missing: X_train[m]=0.0; X_test[m]=0.0
    X_train = X_train[expected]; X_test = X_test[expected]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train); X_test_s = scaler.transform(X_test)
spw = (y_train==0).sum()/(y_train==1).sum()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train years: {train_df["start_year"].min()}-{train_df["start_year"].max()}')
print(f'Test  years: {test_df["start_year"].min()}-{test_df["start_year"].max()}')
print(f'Train: {len(X_train)} | Test: {len(X_test)} | spw: {spw:.2f}')

Train years: 2016-2020
Test  years: 2020-2020
Train: 964 | Test: 242 | spw: 0.38


## XGBoost search

In [2]:
xgb_param_grid = {
    'n_estimators':     [100,200,300],
    'max_depth':        [3,4,5,6],
    'learning_rate':    [0.01,0.05,0.1,0.2],
    'subsample':        [0.6,0.7,0.8,1.0],
    'colsample_bytree': [0.6,0.7,0.8,1.0],
    'min_child_weight': [1,3,5],
    'gamma':            [0,0.1,0.2],
    'scale_pos_weight': [spw, 0.5, 0.637, 1.0],
}

xgb_base = XGBClassifier(random_state=42, eval_metric='auc', verbosity=0)
xgb_search = RandomizedSearchCV(xgb_base, xgb_param_grid,
                                 n_iter=50, scoring='roc_auc',
                                 cv=cv, random_state=42, n_jobs=-1)
xgb_search.fit(X_train_s, y_train)
xgb_best = xgb_search.best_estimator_
y_prob_xgb_t = xgb_best.predict_proba(X_test_s)[:,1]
y_pred_xgb_t = xgb_best.predict(X_test_s)
print('XGB Tuned')
print(f'  Best CV AUC: {xgb_search.best_score_:.4f}')
print(f'  Test AUC   : {roc_auc_score(y_test,y_prob_xgb_t):.4f}')
print(f'  Test Acc   : {accuracy_score(y_test,y_pred_xgb_t)*100:.2f}%')
print(classification_report(y_test, y_pred_xgb_t, target_names=['Low Injury','High Injury']))

XGB Tuned
  Best CV AUC: 0.6372
  Test AUC   : 0.6723
  Test Acc   : 67.36%
              precision    recall  f1-score   support

  Low Injury       0.82      0.19      0.31        93
 High Injury       0.66      0.97      0.79       149

    accuracy                           0.67       242
   macro avg       0.74      0.58      0.55       242
weighted avg       0.72      0.67      0.60       242



## Random Forest search

In [3]:
rf_param_grid = {
    'n_estimators':      [100,200,300],
    'max_depth':         [5,10,15,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf':  [1,2,4],
    'max_features':      ['sqrt','log2'],
    'class_weight':      ['balanced','balanced_subsample'],
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_search = RandomizedSearchCV(rf_base, rf_param_grid,
                                n_iter=50, scoring='roc_auc',
                                cv=cv, random_state=42, n_jobs=-1)
rf_search.fit(X_train_s, y_train)
rf_best = rf_search.best_estimator_
y_prob_rf_t = rf_best.predict_proba(X_test_s)[:,1]
print('RF Tuned')
print(f'  Best CV AUC: {rf_search.best_score_:.4f}')
print(f'  Test AUC   : {roc_auc_score(y_test,y_prob_rf_t):.4f}')

RF Tuned
  Best CV AUC: 0.6278
  Test AUC   : 0.6668


## Logistic Regression baseline (for comparison)

In [4]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
y_prob_lr = lr.predict_proba(X_test_s)[:,1]
print(f'LR baseline AUC: {roc_auc_score(y_test, y_prob_lr):.4f}')

LR baseline AUC: 0.6263


## Side-by-side comparison

Random-split tuned numbers from `notebooks/cycle3_tuning.ipynb`:
- Logistic Regression baseline: 0.6220 (winner)
- XGBoost Tuned: 0.6179
- Random Forest Tuned: ~0.59

In [5]:
results = pd.DataFrame([
    {'Model':'Logistic Regression (Baseline)','Chrono AUC':roc_auc_score(y_test,y_prob_lr),  'Random AUC':0.6220},
    {'Model':'XGBoost Tuned',                  'Chrono AUC':roc_auc_score(y_test,y_prob_xgb_t),'Random AUC':0.6179},
    {'Model':'Random Forest Tuned',            'Chrono AUC':roc_auc_score(y_test,y_prob_rf_t), 'Random AUC':0.5916},
])
results['Delta'] = (results['Chrono AUC']-results['Random AUC']).round(4)
results['Chrono AUC'] = results['Chrono AUC'].round(4)
print(results.to_string(index=False))

                         Model  Chrono AUC  Random AUC  Delta
Logistic Regression (Baseline)      0.6263      0.6220 0.0043
                 XGBoost Tuned      0.6723      0.6179 0.0544
           Random Forest Tuned      0.6668      0.5916 0.0752


## Save best chronological model

Compares LR baseline, XGBoost Tuned and RF Tuned on the chronological test set and saves whichever wins. Overwrites the deployed `models/cycle3_*` artefacts so the API serves the chronologically-trained winner.

Also re-prints the XGBoost best_params (was missing from earlier output).

In [6]:
import joblib, os

print('XGBoost best_params:', xgb_search.best_params_)
print('RF      best_params:', rf_search.best_params_)
print()

lr_auc  = roc_auc_score(y_test, y_prob_lr)
xgb_auc = roc_auc_score(y_test, y_prob_xgb_t)
rf_auc  = roc_auc_score(y_test, y_prob_rf_t)

candidates = {
    'Logistic Regression':  (lr,                        lr_auc),
    'XGBoost Tuned':        (xgb_search.best_estimator_, xgb_auc),
    'Random Forest Tuned':  (rf_search.best_estimator_,  rf_auc),
}
best_name = max(candidates, key=lambda k: candidates[k][1])
best_model, best_auc = candidates[best_name]
print(f'Saving: {best_name}  (test AUC = {best_auc:.4f})')

os.makedirs('../models', exist_ok=True)
joblib.dump(best_model,            '../models/cycle3_best_model.pkl')
joblib.dump(scaler,                '../models/cycle3_scaler.pkl')
joblib.dump(list(X_train.columns), '../models/cycle3_feature_cols.pkl')
print('Saved → ../models/cycle3_best_model.pkl')
print('Saved → ../models/cycle3_scaler.pkl')
print('Saved → ../models/cycle3_feature_cols.pkl')

XGBoost best_params: {'subsample': 0.8, 'scale_pos_weight': 1.0, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0.2, 'colsample_bytree': 0.8}
RF      best_params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 10, 'class_weight': 'balanced'}

Saving: XGBoost Tuned  (test AUC = 0.6723)
Saved → ../models/cycle3_best_model.pkl
Saved → ../models/cycle3_scaler.pkl
Saved → ../models/cycle3_feature_cols.pkl
